In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score,f1_score,roc_curve, roc_auc_score, ConfusionMatrixDisplay, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
import time
import copy
import os
from collections import Counter
from sklearn.metrics import log_loss
import pickle

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

c:\Users\ameli\OneDrive\Studium\TUWien\SS2026\PrivacyInML\Exercise2


In [57]:
def evaluate_true_class_confidence(input_confidence_scores, y_train, y_test):

    true_labels = np.concatenate([y_train, y_test])
    class_order = np.array(['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA'])

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_class_indices = np.array([np.where(class_order == label)[0][0]for label in true_labels])
        true_confidences = scores[np.arange(len(true_labels)),true_class_indices]

        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())


    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [58]:
def evaluate_true_max_confidence(input_confidence_scores, y_train, y_test):

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_confidences = scores.max(axis=1)
        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())

    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [59]:
def evaluate_threshold_attack(
    input_confidence_scores,
    y_train,
    y_test,
    threshold
):

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])
    for key, scores in input_confidence_scores.items():

        # Use the highest predicted probability
        confidence = scores.max(axis=1)

        # Predict membership based on threshold
        attack_labels = np.array([1 if conf > threshold else 0 for conf in confidence])

        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Threshold:", threshold)

        print("Predicted members:",np.sum(attack_labels == 1))
        print("Predicted non-members:",np.sum(attack_labels == 0))
        print("Actual members:",np.sum(membership == 1))
        print("Actual non-members:",np.sum(membership == 0))

        print("Accuracy:",accuracy_score(membership, attack_labels))
        print("Confusion matrix:")

        cm = confusion_matrix(membership,attack_labels)
        print(cm)
        print('=========================================\n')

In [60]:
bean_train = pd.read_csv("datasets/bean_train.csv")
bean_test = pd.read_csv("datasets/bean_test.csv")

X_train = bean_train.drop('Class', axis = 1)
X_test = bean_test.drop('Class', axis = 1)
y_train = bean_train['Class']
y_test = bean_test['Class']

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

----------------------------------------------------------------------------------------------
-------------------------------------- INPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [61]:
with open("input_confidence_scores_bean.pkl", "rb") as f:
    input_confidence_scores = pickle.load(f)

In [62]:
evaluate_true_class_confidence(input_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(input_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.03835133348027331
  Member mean: 0.03719691403379868
  Non-member mean: 0.04296731546088873
  Confidence gap: -0.005770401427090052
  MIA AUC: 0.49711479928645497

Loss:
  Overall mean: 22.142778842259247
  Member mean: 22.169360332344397
  Non-member mean: 22.036491929278306
  Loss gap: -0.13286840306609093
  MIA AUC: 0.49711479928645497

Entropy:
  Overall mean: 0.0
  Member mean: 0.0
  Non-member mean: 0.0
  Entropy gap: 0.0
  MIA AUC: 0.5

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.03835133348027331
  Member mean: 0.03719691403379868
  Non-member mean: 0.04296731546088873
  Confidence gap: -0.005770401427090052
  MIA AUC: 0.49711479928645497

Loss:
  Overall mean: 22.142778842259247
  Member mean: 22.169360332344397
  Non-member mean: 22.036491929278306
  Loss gap: -0.13286840306609093
  MIA AUC: 0.49711479928645497

Entropy:
  Overal

In [63]:
evaluate_threshold_attack(input_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-

----------------------------------------------------------------------------------------------
------------------------------------- OUTPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [64]:
with open("output_confidence_scores_bean.pkl", "rb") as f:
    output_confidence_scores = pickle.load(f)

In [65]:
evaluate_true_class_confidence(output_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(output_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.03835133348027331
  Member mean: 0.03719691403379868
  Non-member mean: 0.04296731546088873
  Confidence gap: -0.005770401427090052
  MIA AUC: 0.49711479928645497

Loss:
  Overall mean: 22.142778842259247
  Member mean: 22.169360332344397
  Non-member mean: 22.036491929278306
  Loss gap: -0.13286840306609093
  MIA AUC: 0.49711479928645497

Entropy:
  Overall mean: 0.0
  Member mean: 0.0
  Non-member mean: 0.0
  Entropy gap: 0.0
  MIA AUC: 0.5

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.03835133348027331
  Member mean: 0.03719691403379868
  Non-member mean: 0.04296731546088873
  Confidence gap: -0.005770401427090052
  MIA AUC: 0.49711479928645497

Loss:
  Overall mean: 22.142778842259247
  Member mean: 22.169360332344397
  Non-member mean: 22.036491929278306
  Loss gap: -0.13286840306609093
  MIA AUC: 0.49711479928645497

Entropy:
  Overal

In [66]:
evaluate_threshold_attack(output_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 13611
Predicted non-members: 0
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.7999412240099919
Confusion matrix:
[[    0  2723]
 [    0 10888]]

-

----------------------------------------------------------------------------------------------
------------------------------------ INTERNAL PERTUBATION ------------------------------------
----------------------------------------------------------------------------------------------

In [67]:
with open("internal_confidence_scores_bean.pkl", "rb") as f:
    internal_confidence_scores = pickle.load(f)

In [68]:
evaluate_true_class_confidence(internal_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(internal_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.11460149829406761
  Member mean: 0.11449645766328523
  Non-member mean: 0.11502150651586664
  Confidence gap: -0.0005250488525814084
  MIA AUC: 0.5070591045123276

Loss:
  Overall mean: 11.239551573028548
  Member mean: 11.165665131227616
  Non-member mean: 11.534988803409947
  Loss gap: 0.36932367218233075
  MIA AUC: 0.5074774460517166

Entropy:
  Overall mean: 1.2169687701036467
  Member mean: 1.2168031698083042
  Non-member mean: 1.2176309280234734
  Entropy gap: 0.0008277582151692275
  MIA AUC: 0.501113160189023

Confidence:
  Minimum: 7.999726423084365e-137
  Maximum: 0.9999999096255034

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.23573999185156458
  Member mean: 0.23518908984208217
  Non-member mean: 0.23794279063204382
  Confidence gap: -0.0027537007899616495
  MIA AUC: 0.4995392947604198

Loss:
  Overall mean: 5.845585683191966
  Member mean: 5.831558105127084
  Non-member 

In [69]:
evaluate_threshold_attack(internal_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 300
Predicted non-members: 13311
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.21240173389170525
Confusion matrix:
[[ 2657    66]
 [10654   234]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 1745
Predicted non-members: 11866
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.2796267724634487
Confusion matrix:
[[2392  331]
 [9474 1414]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 1063
Predicted non-members: 12548
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.24700609800896334
Confusion matrix:
[[ 2511   212]
 [10037   851]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 1219
Predicted non-members: 12392
Actual members: 10888
Actual non-members: 2723
Accuracy: 0.25126735728454924
Confusion matrix:
[[2462  261]
 [9930  95